# Dwarf tomato canopy images: get them into Colab

The 4TU image archive is **one 31 GB zip** (7,420 files). You don't need all of it.
This notebook reads the zip **over the internet** and pulls out only the images you ask for,
shrinking each one and saving it to Google Drive. Nothing big is ever stored in Colab.

| Camera | Compartment | Team |
|---|---|---|
| cam_18 | 3.08 | Trigger |
| cam_19 | 3.07 | IDEAS |
| cam_1 | 3.06 | Reference |
| cam_20 | 3.03 | Agrifusion |
| cam_21 | 3.02 | MuGrow |
| camir_27 | 3.01 | Tomatonuts |

Each camera took images at **04, 08, 12 and 16 h** every day: an RGB photo (3840x2160, ~15 MB),
a depth image (~1 MB) and two infrared images. Use **12:00 RGB**: at 04:00 the purple grow lamps colour everything pink.

In [ ]:
!pip -q install remotezip
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re, io, os
import pandas as pd
from remotezip import RemoteZip

URL = 'https://data.4tu.nl/file/fa102772-32db-4b30-bace-12f2016722ce/818e2966-7015-432d-aa7c-1b9158c011cf'
OUT = '/content/drive/MyDrive/tomato_images'   # where the small images go
os.makedirs(OUT, exist_ok=True)

CAM_TEAM = {'cam_18': 'trigger', 'cam_19': 'ideas', 'cam_1': 'reference',
            'cam_20': 'agrifusion', 'cam_21': 'mugrow', 'camir_27': 'tomatonuts'}

# Read only the zip's table of contents (a few hundred KB), not the 31 GB
with RemoteZip(URL) as z:
    names = z.namelist()

pat = re.compile(r'dataset4tu/(\w+)/\1_(\d{4})_(\d\d)_(\d\d)_(\d\d)_\d\d_\d\d_(\w+)\.png')
rows = [dict(file=n, cam=m[1], team=CAM_TEAM[m[1]], date=f'{m[2]}-{m[3]}-{m[4]}', hour=int(m[5]), kind=m[6])
        for n in names if (m := pat.match(n))]
index = pd.DataFrame(rows)
print(len(index), 'images')
index.groupby(['team', 'kind']).size().unstack()

## Choose what to download
Start small. The default is **12:00 RGB for every camera and every day**: 464 photos,
about 7.4 GB transferred but only ~500 MB saved, because each photo is shrunk to 960x540.
Change the filter to grab fewer (e.g. one team) or more (e.g. add `'depth'`).

In [ ]:
pick = index[(index.kind == 'rgb') & (index.hour == 12)]
# pick = pick[pick.team == 'reference']          # uncomment to test with one team first
print(len(pick), 'files to fetch')

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import threading
from tqdm.auto import tqdm

local = threading.local()
def zipfile_for_thread():
    if not hasattr(local, 'z'):
        local.z = RemoteZip(URL)  # one connection per thread
    return local.z

SIZE = (960, 540)  # shortcut: 4x smaller than original; use full size only if you need fine detail

def fetch(row):
    dst = f"{OUT}/{row.team}_{row.date}_{row.hour:02d}_{row.kind}.png"
    if os.path.exists(dst):            # lets you re-run after a disconnect
        return dst
    img = Image.open(io.BytesIO(zipfile_for_thread().read(row.file)))
    if row.kind == 'rgb':
        img = img.convert('RGB').resize(SIZE, Image.BILINEAR)
    else:
        img = img.resize(SIZE, Image.NEAREST)  # depth: never blend distance values
    img.save(dst)
    return dst

with ThreadPoolExecutor(8) as ex:
    done = list(tqdm(ex.map(fetch, pick.itertuples()), total=len(pick)))
print('saved', len(done), 'images to', OUT)

In [ ]:
# Look at one compartment over time
import matplotlib.pyplot as plt
team = 'reference'
days = ['2024-10-07', '2024-10-22', '2024-11-05']
fig, ax = plt.subplots(1, len(days), figsize=(18, 4))
for a, d in zip(ax, days):
    a.imshow(Image.open(f'{OUT}/{team}_{d}_12_rgb.png')); a.set_title(f'{team} {d}'); a.axis('off')

## A first ripeness signal: share of red fruit pixels
A rough starting point, not a final method: count pixels whose colour is tomato-red/orange
versus all fruit-coloured pixels. Thresholds are in HSV colour space and should be checked by eye on a few images.

In [ ]:
import numpy as np

def red_fraction(path):
    # PIL HSV: hue, saturation, value all on a 0-255 scale
    h, s, v = np.moveaxis(np.asarray(Image.open(path).convert('HSV')), -1, 0).astype(int)
    bright = (s > 90) & (v > 80)
    red = bright & ((h < 17) | (h > 234))           # red / deep orange
    return red.mean() * 100                          # % of image that is red fruit

rows = []
for f in sorted(os.listdir(OUT)):
    if f.endswith('_12_rgb.png'):
        team, date = f.split('_')[:2]
        rows.append(dict(team=team, date=pd.to_datetime(date), red_pct=red_fraction(f'{OUT}/{f}')))
red = pd.DataFrame(rows)
red.to_csv(f'{OUT}/red_fraction_daily.csv', index=False)
red.pivot(index='date', columns='team', values='red_pct').plot(figsize=(12, 4), ylabel='% red pixels')

## If you'd rather download the whole zip
Colab has roughly 80 GB of free disk, so it fits, but it takes a while and is lost when the session ends:
```
!wget -c -O /content/canopy.zip "<URL above>"
!unzip -l /content/canopy.zip | head          # list
!unzip /content/canopy.zip "dataset4tu/cam_1/*_12_*_rgb.png" -d /content/imgs   # extract a subset
```
Google Drive's free 15 GB can't hold the full zip, so keep only the images or numbers you need there.